In [1]:
import sys
import time
import requests
import pandas as pd

def get_game_reviews_details(game_ids, reviews_path_output, max_reviews_per_game=2000):
  # Wczytanie istniejących recenzji, aby móc je później połączyć
  try:
    df_reviews = pd.read_csv(reviews_path_output)
  except Exception:
    sys.exit('Ścieżka nie istnieje, spróbuj ponownie po utworzeniu pliku.')

  headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
  }
  
  # Parametry API Steam dla pobierania samych recenzji
  base_params = {
    'json': 1,
    'filter': 'recent',   # Pobieramy najnowsze, skoro chcemy odświeżać dane
    'language': 'polish',   # Interesują nas tylko polskie opinie
    'day_range': 9223372036854775807, # Maksymalny zakres czasu (brak limitu dni)
    'review_type': 'all',
    'purchase_type': 'all',
    'num_per_page': 100   # Maksymalna dopuszczalna liczba recenzji na jedną paczkę
  }

  all_new_reviews = []
  rate_limit_hits = 0
  max_rate_hits = 5
  initial_sleep = 2
  
  total_games = len(game_ids)

  for g_idx, game_id in enumerate(game_ids, 1):
    print(f'\n=== [{g_idx}/{total_games}] Rozpoczęcie pobierania recenzji dla Gry ID: {game_id}')
    
    cursor = '*'  # Początkowy cursor dla nowej gry
    reviews_fetched_for_game = 0
    
    while reviews_fetched_for_game < max_reviews_per_game:
      url = f'https://store.steampowered.com/appreviews/{game_id}'
      
      # Kopiujemy parametry bazy i dorzucamy aktualny cursor
      params = base_params.copy()
      params['cursor'] = cursor
      
      try:
        response = requests.get(url, headers=headers, params=params, timeout=14)

        # Obsługa Rate Limit (HTTP 429)
        if response.status_code == 429:
          if rate_limit_hits < max_rate_hits:
            rate_limit_hits += 1
            sleep_time = initial_sleep * (2 ** rate_limit_hits)
            print(f'STOP! Rate limit (429). Zasypiam na {sleep_time}s...')
            time.sleep(sleep_time)
            continue  # Powtarzamy obróbkę tego samego kursora
          else:
            print('Przekroczono limity zapytań. Awaryjny zapis zebranych danych...')
            break # Przerywamy pętlę pobierania i przechodzimy do zapisu tego co mamy

        if response.status_code == 200:
          data = response.json()
          
          if data.get('success') != 1:
            print(f'Steam zwrócił success=0 dla ID {game_id}. Przerywam tę grę.')
            break
          
          reviews = data.get('reviews', [])
          if not reviews:
            print(f'Brak kolejnych recenzji dla gry {game_id}.')
            break
          
          # Parsowanie otrzymanych recenzji
          for rev in reviews:
            all_new_reviews.append({
              'steam_id': game_id,
              'recommendationid': str(rev.get('recommendationid')),
              'author_steamid': rev.get('author', {}).get('steamid'),
              'playtime_forever': rev.get('author', {}).get('playtime_forever'),
              'review': rev.get('review', ''),
              'voted_up': rev.get('voted_up'), # True = pozytywna, False = negatywna
              'votes_up': rev.get('votes_up', 0),
              'timestamp_created': rev.get('timestamp_created')
            })
          
          reviews_fetched_for_game += len(reviews)
          print(f'-> Pobrano {reviews_fetched_for_game}/{max_reviews_per_game} recenzji...')
          
          # Pobranie nowego cursora do kolejnej paczki
          next_cursor = data.get('cursor')
          if not next_cursor or next_cursor == cursor:
            break
          cursor = next_cursor
          
          # Reset licznika rate limitów po udanym zapytaniu
          rate_limit_hits = max(0, rate_limit_hits - 1)
          
        else:
          print(f'Błąd HTTP {response.status_code} dla ID {game_id}. Przerywam tę grę.')
          break

      except Exception as e:
        print(f'Nieoczekiwany błąd przy pobieraniu recenzji dla ID {game_id}: {e}')
        break
      
      # Szybki oddech dla API między stronami (Steam jest bardzo wrażliwy na pobieranie recenzji)
      time.sleep(0.5)
      
    if rate_limit_hits >= max_rate_hits:
      break

  # --- PROCES ŁĄCZENIA I ZAPISU DANYCH ---
  if all_new_reviews:
    df_new_reviews = pd.DataFrame(all_new_reviews)
    
    if not df_reviews.empty:
      # Usuwamy z historycznego pliku stare wiersze dla gier, które właśnie pobieraliśmy,
      # ponieważ pobraliśmy dla nich najświeższe pakiety danych.
      df_old_filtered = df_reviews[~df_reviews['steam_id'].isin(game_ids)]
      
      # Łączymy stare przefiltrowane dane z nowymi danymi
      df_final = pd.concat([df_new_reviews, df_old_filtered], ignore_index=True)
    else:
      df_final = df_new_reviews
      
    # Zabezpieczenie: usuwamy duplikaty po unikalnym ID recenzji (na wypadek nałożeń)
    df_final = df_final.drop_duplicates(subset=['recommendationid'])
    
    # Nadpisujemy plik CSV ('w')
    df_final.to_csv(reviews_path_output, mode='w', index=False)
    print(f'\nSukces! Plik zaktualizowany. Łączna liczba recenzji w bazie: {len(df_final)}')
    return df_final
  else:
    print('\nNie pobrano żadnych nowych recenzji. Plik pozostaje bez zmian.')
    return df_reviews

In [6]:
# create empty file

new_df = pd.DataFrame({}, columns= 
                      ['steam_id',
              'recommendationid',
              'author_steamid',
              'playtime_forever',
              'review',
              'voted_up',
              'votes_up',
              'timestamp_created',
            ])
print(new_df)
reviews_path_output = '../data/reviews.csv'
new_df.to_csv(reviews_path_output, mode='w', index=False)

Empty DataFrame
Columns: [steam_id, recommendationid, author_steamid, playtime_forever, review, voted_up, votes_up, timestamp_created]
Index: []


In [7]:

game_ids = ['444940', '2357570']
get_game_reviews_details(game_ids, reviews_path_output, max_reviews_per_game=2000)


=== [1/2] Rozpoczęcie pobierania recenzji dla Gry ID: 444940
-> Pobrano 7/2000 recenzji...
Brak kolejnych recenzji dla gry 444940.

=== [2/2] Rozpoczęcie pobierania recenzji dla Gry ID: 2357570
-> Pobrano 100/2000 recenzji...
-> Pobrano 200/2000 recenzji...
-> Pobrano 300/2000 recenzji...
-> Pobrano 400/2000 recenzji...
-> Pobrano 500/2000 recenzji...
-> Pobrano 600/2000 recenzji...
-> Pobrano 700/2000 recenzji...
-> Pobrano 800/2000 recenzji...
-> Pobrano 900/2000 recenzji...
-> Pobrano 1000/2000 recenzji...
-> Pobrano 1100/2000 recenzji...
-> Pobrano 1200/2000 recenzji...
-> Pobrano 1300/2000 recenzji...
-> Pobrano 1400/2000 recenzji...
-> Pobrano 1500/2000 recenzji...
-> Pobrano 1600/2000 recenzji...
-> Pobrano 1700/2000 recenzji...
-> Pobrano 1800/2000 recenzji...
-> Pobrano 1900/2000 recenzji...
-> Pobrano 2000/2000 recenzji...

Sukces! Plik zaktualizowany. Łączna liczba recenzji w bazie: 2007


,steam_id,recommendationid,author_steamid,playtime_forever,review,voted_up,votes_up,timestamp_created
0,444940,207711335,76561198096017729,168,el kaka,False,4,1761566872
1,444940,206952881,76561198272902337,25,"The game crashed mid-game. What's worse, I was...",False,0,1760732364
2,444940,187429151,76561199823379718,29,ll,True,0,1738981642
3,444940,166106052,76561198848912404,10,broken,False,1,1716736146
4,444940,82337143,76561198241402054,186,Polecam.,True,0,1607942677
...,...,...,...,...,...,...,...,...
2002,2357570,166104402,76561198997878707,79,blizzard śmierdzi,False,0,1716734431
2003,2357570,166062496,76561198082955414,21,♥♥♥♥ blizzard ♥♥♥♥♥♥s,False,0,1716674888
2004,2357570,166051383,76561198328164986,1947,skasujcie fare i sombre to bedzie zajebiscie,False,0,1716661782
2005,2357570,166038290,76561198317749245,709,jest lepiej,True,0,1716650087
